# ヒートマップ生成系

In [ ]:
import os
from pathlib import Path
import sys
from matplotlib import pyplot as plt
sys.path.append('../')  # srcディレクトリをimportできるようにする
from src.my_app.core.MyDataset.FineTuningDataset_v1 import FineTuningDataset_v1
from src.my_app.core.MyDataset import FineTuningDataset_v0
IMAGES_DIR = Path('../../kuzushiji-recognition/char_sep_datas')  # 画像ディレクトリ
GT_JSON_PATH = Path('../../kuzushiji-recognition/char_sep_datas/gt_json.json')    # アノテーションJSON (未使用でもロード例)


In [11]:
os.listdir('../../kuzushiji-recognition/char_sep_datas')


['gt_json.json', '200021869', '.DS_Store', '200021637']

## データセット定義

In [6]:
from torch.utils.data import DataLoader
test_doc_id_list = [
    '200021637',
    '100249371',
    '100249537',
    '200005598',
    '200014740',
    '200020019',
    '200021712',
    '200021869'
]
train_dataset = FineTuningDataset_v1(
    
    test_doc_id=test_doc_id_list,
    test_mode=False,
    images_dir=IMAGES_DIR, 
    json_path=GT_JSON_PATH,
    precompute_gt=True,
    target_width=100,
    )
test_dataset = FineTuningDataset_v1(
    test_doc_id=test_doc_id_list,
    test_mode=True,
    images_dir=IMAGES_DIR,
    json_path=GT_JSON_PATH,
    precompute_gt=True,
    target_width=100,
)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)


AssertionError: 画像ディレクトリが存在しません: ../kuzushiji-recognition/char_sep_datas

## モデル定義

In [2]:
from src.my_app import UNet, create_optimized_dataloader
from src.my_app import UNet, create_optimized_dataloader
import torch

check_point_version = '1.1'
checkpoint_dir = "../.checkpoints"
checkpoint_path = os.path.join(checkpoint_dir, f"latest_checkpoint_V{check_point_version}.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(3, 4).to(device)
def weighted_mse_loss(pred, target, thresh=0.01, pos_weight=20, use_continuous=True, eps=1e-8):
    """
    pred: (B,C,H,W)
    target: (B,C,H,W)
    thresh: 正例二値化の閾値 (use_continuous=False のとき)
    pos_weight: 正例に掛ける重み (または連続重み係数 α)
    use_continuous: Trueなら w = 1 + pos_weight * target（ターゲット値で連続的に重み付け）
    """
    if use_continuous:
        w = 1.0 + pos_weight * target   # target が 0～1 と想定
    else:
        pos = (target > thresh).float()
        w = 1.0 + (pos_weight - 1.0) * pos
    diff2 = (pred - target) ** 2
    loss = (w * diff2).sum() / (w.sum().clamp_min(eps))
    return loss
criterion = weighted_mse_loss # 回帰問題なのでMSE損失を使用 
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
os.makedirs(checkpoint_dir, exist_ok=True)

# 最良のモデルを追跡するための変数
best_test_loss = float('inf')
start_epoch = 0

# 損失の履歴を保存するリストを初期化
train_loss_history = []
test_loss_history = []

# チェックポイントの読み込み（存在する場合）
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = 0
    best_test_loss = checkpoint.get('best_test_loss', float('inf'))
    train_loss_history = checkpoint.get('train_loss_history', train_loss_history)
    test_loss_history  = checkpoint.get('test_loss_history',  test_loss_history)
    print(f"[main]: チェックポイントを読み込みました（エポック {start_epoch}）")
else:
    start_epoch = 0
    best_test_loss = float('inf')
# 以降は train_loss_history / test_loss_history を再初期化しない

def crop_labels_to_match(labels_to_crop, target_tensor):
    target_h, target_w = target_tensor.shape[2:]
    source_h, source_w = labels_to_crop.shape[2:]
    delta_h = (source_h - target_h) // 2
    delta_w = (source_w - target_w) // 2
    return labels_to_crop[:, :, delta_h:delta_h + target_h, delta_w:delta_w + target_w]

ModuleNotFoundError: No module named 'src'

## ヒートマップを推測

In [ ]:
from tqdm import tqdm
epoch = 0
num_epochs = 1

checkpoint_dir = "../checkpoints_finetuning"
check_point_version = '1.0'
checkpoint_path = os.path.join(checkpoint_dir, f"latest_checkpoint_V{check_point_version}.pth")

# チェックポイントの読み込み（存在する場合）
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    # optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = 0
    best_test_loss = checkpoint.get('best_test_loss', float('inf'))
    train_loss_history = checkpoint.get('train_loss_history', train_loss_history)
    test_loss_history  = checkpoint.get('test_loss_history',  test_loss_history)
    print(f"[main]: チェックポイントを読み込みました（エポック {start_epoch}）")
else:
    start_epoch = 0
    best_test_loss = float('inf')
    print(f"[main]: チェックポイントが見つかりません。新しいモデルで開始します。")
# 以降は train_loss_history / test_loss_history を再初期化しない

test_bar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
for batch in test_bar:
    # バッチは (imgs, masks, file_id) 想定
    if len(batch) == 3:
        imgs = batch[0]
        masks = batch[1]
        file_ids = batch[2]
    elif len(batch) == 2:
        imgs = batch[0]
        masks = batch[1]
        file_ids = None
    pred = model(imgs.to(device))
    # for i in range(4):
    #     plt.imshow(masks[0][i].cpu().detach())
    #     plt.show()
    cropped_imgs = crop_labels_to_match(imgs, pred)
    plt.imshow(cropped_imgs[0].cpu().detach().permute(1,2,0))
    plt.show()
    for j in range(4):
        plt.imshow(pred[0][j].cpu().detach())
        plt.show()
    for i in range(4):
        train_dataset.show_combine_3chanel_and_1chanel(image_A=cropped_imgs[0], image_B=pred[0][i].cpu().detach())

# 新しい取り組み - 値を推測 - 

In [ ]:
pred.shape


torch.Size([1, 4, 128, 96])